# RAG-Based Sustainability Enhancement for Business Model Canvas

## Objective

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline designed to improve a Business Model Canvas (BMC) by integrating **external sustainability knowledge**.

The goal is to move beyond basic text generation by grounding the model’s responses in **reliable, domain-specific information** related to sustainability and Sustainable Development Goals (SDGs).

## Why this is useful

Traditional language models generate responses based only on what they learned during training.  
This can lead to:
- generic suggestions
- lack of domain-specific insights
- hallucinations (incorrect information)

To solve this, we use **RAG**, which allows the model to:
- retrieve relevant sustainability knowledge from external documents
- use this knowledge to generate more accurate and meaningful improvements
- align business strategies with sustainability principles

## What this notebook does

This notebook builds a complete pipeline that:

1. Loads sustainability-related documents  
2. Splits them into meaningful chunks  
3. Converts them into embeddings (vector representations)  
4. Stores them in a FAISS index for fast retrieval  
5. Retrieves relevant knowledge based on a BMC input  
6. Filters results using predicted SDGs  
7. Constructs a structured prompt  
8. Uses an LLM to generate improved BMC blocks  

## Final Output

The system generates **enhanced versions of the 9 Business Model Canvas blocks**, with sustainability-driven recommendations based on retrieved knowledge.

## Key Technologies

- Sentence Transformers (for embeddings)
- FAISS (for similarity search)
- Large Language Model (Qwen)
- RAG (Retrieval-Augmented Generation)

---
 In short:  
This notebook transforms a simple BMC into a **sustainability-aware business model** .

## 2. Environment Setup

We install and import the required libraries for embeddings, retrieval, and text generation.

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.1 MB/s eta 0:00:00:00:0100:01


## 3. Loading Documents

We load sustainability-related documents that will serve as external knowledge for the system.

In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

DOCS_DIR = "/kaggle/input/datasets/jihenedorgham/sustainability-docs/sustainability_docs2"   
OUTPUT_DIR = "/kaggle/working/rag_index"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
documents = []

for filename in os.listdir(DOCS_DIR):
    if filename.endswith(".txt"):
        file_path = os.path.join(DOCS_DIR, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "source": filename,
            "text": text
        })

print("Documents loaded:", len(documents))

for doc in documents:
    print("-", doc["source"], "| characters:", len(doc["text"]))

Documents loaded: 6
- sustainable_bmc.txt | characters: 2930
- green_business.txt | characters: 1712
- circular_economy.txt | characters: 1332
- sdg.txt | characters: 5404
- carbon.txt | characters: 1527
- green_logistics.txt | characters: 1565


## 4. Document Chunking

The documents are split into smaller chunks to make retrieval more efficient and precise.

In [4]:
import re

def is_doc_heading(line):
    line = line.strip()
    if not line:
        return False

    patterns = [
        r"^SDG\s+\d+\s+",
        r"^Sustainable\s+",
        r"^Energy\s+",
        r"^Waste\s+",
        r"^Eco-Design\s+",
        r"^Reducing\s+",
        r"^Route\s+",
        r"^Local\s+",
        r"^Low-Carbon\s+",
        r"^Carbon\s+",
        r"^Strategies\s+",
        r"^Impact\s+",
        r"^Reuse\s+",
        r"^Extending\s+",
        r"^Circular\s+",
    ]

    return any(re.match(p, line) for p in patterns)


def chunk_by_plain_headings(text, source):
    lines = text.splitlines()
    sections = []
    current = []

    for line in lines:
        clean_line = line.strip()

        if is_doc_heading(clean_line):
            if current:
                section = "\n".join(current).strip()
                if len(section) > 80:
                    sections.append(section)
            current = [clean_line]
        else:
            if clean_line:
                current.append(clean_line)

    if current:
        section = "\n".join(current).strip()
        if len(section) > 80:
            sections.append(section)

    return [{"source": source, "text": s} for s in sections]

## 5. Building Knowledge Base

We create a list of text chunks with their sources to form the knowledge base.

In [5]:
chunks = []

for doc in documents:
    doc_chunks = chunk_by_plain_headings(doc["text"], doc["source"])
    print(doc["source"], "→", len(doc_chunks), "chunks")

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": len(chunks),
            "source": chunk["source"],
            "local_chunk_id": i,
            "text": chunk["text"]
        })

print("Total chunks:", len(chunks))

for c in chunks[:5]:
    print("\nSOURCE:", c["source"])
    print(c["text"][:500])

sustainable_bmc.txt → 9 chunks
green_business.txt → 4 chunks
circular_economy.txt → 4 chunks
sdg.txt → 17 chunks
carbon.txt → 5 chunks
green_logistics.txt → 4 chunks
Total chunks: 43

SOURCE: sustainable_bmc.txt
Sustainable Value Proposition in Business Models
Summary: A sustainable value proposition provides environmental benefits alongside functional value.
A business can offer products that reduce energy consumption or emissions. A business can highlight environmental impact as a key differentiator. After defining the value proposition, a business should quantify sustainability benefits.

SOURCE: sustainable_bmc.txt
Sustainable Customer Segments and Targeting
Summary: Customer segments should include sustainability-oriented users.
A business can target eco-conscious consumers and organizations with sustainability requirements. A business can align offerings with ESG expectations. After identifying segments, a business should adapt communication strategies.

SOURCE: sustainable_bmc.t

## 6. Embedding Generation

Each chunk is converted into a vector representation using a Sentence Transformer model.

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (43, 384)


## 7. Vector Index Creation

We store all embeddings in a FAISS index to enable fast similarity search.

In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)

FAISS index created.
Number of vectors: 43


## 8. Saving Index

The index and metadata are saved for reuse without recomputing everything.

In [8]:
faiss.write_index(index, f"{OUTPUT_DIR}/sustainability_faiss.index")

with open(f"{OUTPUT_DIR}/chunks_metadata.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Saved FAISS index and metadata to:", OUTPUT_DIR)

Saved FAISS index and metadata to: /kaggle/working/rag_index


## 9. Retrieval Function

We define a function that retrieves the most relevant chunks based on a query.

In [9]:
def retrieve_context(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        chunk = chunks[idx]
        results.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "distance": float(distance),
            "text": chunk["text"]
        })

    return results

## 10. SDG Filtering

Retrieved chunks are filtered to keep only those aligned with predicted sustainability goals.

In [10]:
def filter_retrieved_chunks_by_predicted_sdgs(retrieved_chunks, predicted_sdgs):
    allowed_numbers = []
    for sdg in predicted_sdgs:
        # extracts number from "SDG 13: Climate Action"
        number = sdg.split(":")[0].replace("SDG", "").strip()
        allowed_numbers.append(number)

    filtered = []

    for chunk in retrieved_chunks:
        text = chunk["text"]

        # If chunk is an SDG chunk, keep it only if it matches predicted SDGs
        if text.startswith("SDG "):
            keep = any(text.startswith(f"SDG {num} ") for num in allowed_numbers)
            if keep:
                filtered.append(chunk)
        else:
            filtered.append(chunk)

    return filtered

## 10. Automatic SDG / ODD Classification

Instead of manually writing the predicted SDGs, this section loads the trained SDG classifier and uses it to predict the most relevant Sustainable Development Goals directly from the Business Model Canvas text.

This makes the final pipeline more automatic:

**BMC text → SDG classifier → predicted SDGs → RAG retrieval → Qwen generation**


In [11]:
# ============================================================
# SDG CLASSIFIER INTEGRATION
# ============================================================

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# IMPORTANT:
# Put here the path of your exported fine-tuned SDG classifier.
# The folder must contain files such as: config.json, tokenizer files,
# and model weights (.safetensors or pytorch_model.bin).
# ------------------------------------------------------------
SDG_CLASSIFIER_CANDIDATE_PATHS = [
    "/kaggle/input/datasets/jihenedorgham/sdg-classif/best_sdg_distilbert",
]


def find_existing_sdg_model_path(candidate_paths):
    for path in candidate_paths:
        if os.path.isdir(path) and os.path.exists(os.path.join(path, "config.json")):
            return path
    return None


SDG_MODEL_PATH = find_existing_sdg_model_path(SDG_CLASSIFIER_CANDIDATE_PATHS)

if SDG_MODEL_PATH is None:
    raise FileNotFoundError(
        "SDG classifier not found. Please upload/add your exported folder "
        "best_sdg_distilbert as a Kaggle dataset, then update "
        "SDG_CLASSIFIER_CANDIDATE_PATHS with the correct path."
    )

print("Loading SDG classifier from:", SDG_MODEL_PATH)

sdg_tokenizer = AutoTokenizer.from_pretrained(SDG_MODEL_PATH)
sdg_classifier = AutoModelForSequenceClassification.from_pretrained(SDG_MODEL_PATH)
sdg_classifier.to(DEVICE)
sdg_classifier.eval()

sdg_names = {
    1: "No Poverty",
    2: "Zero Hunger",
    3: "Good Health and Well-being",
    4: "Quality Education",
    5: "Gender Equality",
    6: "Clean Water and Sanitation",
    7: "Affordable and Clean Energy",
    8: "Decent Work and Economic Growth",
    9: "Industry, Innovation and Infrastructure",
    10: "Reduced Inequalities",
    11: "Sustainable Cities and Communities",
    12: "Responsible Consumption and Production",
    13: "Climate Action",
    14: "Life Below Water",
    15: "Life on Land",
    16: "Peace, Justice and Strong Institutions",
    17: "Partnerships for the Goals"
}


def predict_sdgs_from_text(text, top_k=3, max_length=256):
    """
    Predict the most relevant SDGs from a BMC/startup text.

    Returns a list formatted like:
    ['SDG 13: Climate Action', 'SDG 12: Responsible Consumption and Production']
    """
    inputs = sdg_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = sdg_classifier(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).squeeze()

    top_k = min(top_k, len(sdg_names))
    top_scores, top_indices = torch.topk(probabilities, k=top_k)

    predicted_sdgs = []
    for score, index in zip(top_scores, top_indices):
        sdg_number = int(index.item()) + 1   # labels are 0-based, SDGs are 1-based
        sdg_label = f"SDG {sdg_number}: {sdg_names[sdg_number]}"
        predicted_sdgs.append(sdg_label)
        print(f"{sdg_label} → confidence: {float(score):.4f}")

    return predicted_sdgs


Loading SDG classifier from: /kaggle/input/datasets/jihenedorgham/sdg-classif/best_sdg_distilbert


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## 11. Prompt Construction

We build a structured prompt combining the BMC, automatically predicted SDGs, and retrieved sustainability knowledge.

In [12]:
def build_rag_prompt(bmc_text, predicted_sdgs, retrieved_chunks):
    context = "\n\n".join([
        f"[CHUNK {i+1} | SOURCE FILE: {chunk['source']}]\n{chunk['text']}"
        for i, chunk in enumerate(retrieved_chunks)
    ])

    predicted_sdgs_text = "\n".join([f"- {sdg}" for sdg in predicted_sdgs])

    prompt = f"""
You are a sustainable business model expert.

You must improve the BMC using ONLY the retrieved knowledge below.

RETRIEVED KNOWLEDGE:
{context}

BUSINESS MODEL CANVAS:
{bmc_text}

PREDICTED SDGs:
{predicted_sdgs_text}

TASK:
Generate sustainable improvements for EXACTLY the 9 BMC blocks below, in this exact order:
1. Customer Segments
2. Value Proposition
3. Channels
4. Customer Relationships
5. Revenue Streams
6. Key Resources
7. Key Activities
8. Key Partnerships
9. Cost Structure

For each block, use exactly this format:

#### Block X: Block Name
Current weakness:
Sustainable improvement:
Retrieved knowledge used: mention ONLY the source file name from the retrieved knowledge, such as sustainable_bmc.txt, green_logistics.txt, carbon.txt, sdg.txt, green_business.txt, or circular_economy.txt.
Expected environmental impact:
Linked SDG:
- Existing SDG: choose one SDG from the predicted SDGs if the block already supports it.
- Recommended SDG: suggest one new SDG only if the improvement clearly helps the startup target it.
- Always write the full SDG name, for example: SDG 12: Responsible Consumption and Production.
- Do not write placeholders such as "Sustainable SDG", "e.g.", or "not applicable".
STRICT RULES:
- Do not invent sources.
- Do NOT WRITE "not directly applicable".
- Every block MUST have at least one SDG (existing or recommended)
- Recommended SDGs must be justified clearly
- Retrieved knowledge must specify the exact section (not only file name)
- Avoid generic business suggestions; always connect to sustainability or retrieved knowledge
- Do not add external links.
- Do not repeat blocks.
- Stop immediately after Block 9: Cost Structure.
- If both an Existing SDG and Recommended SDG are relevant, include both.
"""
    return prompt

## 12. Language Model

We load a language model to generate sustainability improvements.

In [13]:
import torch
torch.cuda.empty_cache()

In [14]:
# ============================================================
# LOAD QWEN LLM
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

llm_model.eval()

print("Qwen LLM loaded.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen LLM loaded.


## 13. Text Generation

We generate improved BMC blocks using the constructed prompt.

In [15]:
# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_with_qwen(prompt, max_new_tokens=800):
    messages = [
        {
            "role": "system",
            "content": "You are a sustainability and business model expert."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    response = llm_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response

## 14. Example Input and Automatic SDG Prediction

We define a sample Business Model Canvas, then the SDG classifier automatically predicts the most relevant SDGs.

These predicted SDGs are then used to guide retrieval and generation.

In [16]:
sample_bmc = """
Customer Segments: small online stores and local retailers.
Value Proposition: fast delivery service for businesses.
Channels: mobile app and delivery fleet.
Customer Relationships: customer support and delivery tracking.
Revenue Streams: delivery fees and monthly subscriptions.
Key Resources: vehicles, drivers, logistics software.
Key Activities: package pickup, route planning, delivery.
Key Partnerships: retailers and transport suppliers.
Cost Structure: fuel costs, vehicle maintenance, driver salaries.
"""

# Automatically predict SDGs from the BMC instead of writing them manually
predicted_sdgs = predict_sdgs_from_text(sample_bmc, top_k=3)

query = sample_bmc + " sustainability carbon emissions logistics route optimization energy efficiency waste reduction carbon tax " + " ".join(predicted_sdgs)

retrieved_chunks = retrieve_context(query, top_k=10)
retrieved_chunks = filter_retrieved_chunks_by_predicted_sdgs(retrieved_chunks, predicted_sdgs)
retrieved_chunks = retrieved_chunks[:7]

prompt = build_rag_prompt(sample_bmc, predicted_sdgs, retrieved_chunks)

print("Predicted SDGs:")
for sdg in predicted_sdgs:
    print("-", sdg)

print("\nPROMPT READY")
print("=" * 80)
print(prompt)


SDG 9: Industry, Innovation and Infrastructure → confidence: 0.8259
SDG 11: Sustainable Cities and Communities → confidence: 0.0614
SDG 12: Responsible Consumption and Production → confidence: 0.0289
Predicted SDGs:
- SDG 9: Industry, Innovation and Infrastructure
- SDG 11: Sustainable Cities and Communities
- SDG 12: Responsible Consumption and Production

PROMPT READY

You are a sustainable business model expert.

You must improve the BMC using ONLY the retrieved knowledge below.

RETRIEVED KNOWLEDGE:
[CHUNK 1 | SOURCE FILE: sustainable_bmc.txt]
Sustainable Customer Segments and Targeting
Summary: Customer segments should include sustainability-oriented users.
A business can target eco-conscious consumers and organizations with sustainability requirements. A business can align offerings with ESG expectations. After identifying segments, a business should adapt communication strategies.

[CHUNK 2 | SOURCE FILE: green_logistics.txt]
Reducing Transport Emissions in Supply Chains
Summary

In [17]:
# ============================================================
# TEST RAG + QWEN WITH AUTOMATIC SDG PREDICTION
# ============================================================

rag_response = generate_with_qwen(prompt, max_new_tokens=1500)
print(rag_response)


#### Block 1: Customer Segments
Current weakness:
Small online stores and local retailers may not prioritize sustainability initiatives due to lower visibility and resources dedicated to these efforts.
Sustainable improvement:
Align the value proposition with sustainability goals by offering discounts or special promotions for customers who make purchases with environmentally conscious practices.
Retrieved knowledge used: sustainable_bmc.txt, green_business.txt
Expected environmental impact:
Increased awareness among customers about their purchasing choices, potentially leading to more sustainable consumption habits.
Linked SDG:
- Existing SDG: SDG 12: Responsible Consumption and Production
- Recommended SDG: SDG 14: Life Below Water
Justification:
By focusing on responsible consumption practices, startups can encourage customers to adopt more sustainable behaviors related to water conservation and marine life protection.

#### Block 2: Value Proposition
Current weakness:
The value pro

In [18]:
def run_full_rag_pipeline(bmc_text, top_k_sdgs=3, top_k_chunks=10):
    """
    Full automatic pipeline:
    1. Predict SDGs from the BMC text
    2. Retrieve sustainability knowledge
    3. Filter retrieved SDG chunks according to predicted SDGs
    4. Build the RAG prompt
    5. Generate sustainable BMC improvements with Qwen
    """

    # Step 1 — Predict SDGs automatically
    predicted_sdgs = predict_sdgs_from_text(bmc_text, top_k=top_k_sdgs)

    # Step 2 — Build retrieval query
    query = (
        bmc_text
        + " sustainability carbon emissions logistics route optimization energy efficiency waste reduction circular economy carbon tax "
        + " ".join(predicted_sdgs)
    )

    # Step 3 — Retrieve and filter context
    retrieved_chunks = retrieve_context(query, top_k=top_k_chunks)
    retrieved_chunks = filter_retrieved_chunks_by_predicted_sdgs(retrieved_chunks, predicted_sdgs)
    retrieved_chunks = retrieved_chunks[:7]

    # Step 4 — Build prompt
    prompt = build_rag_prompt(bmc_text, predicted_sdgs, retrieved_chunks)

    # Step 5 — Generate answer
    response = generate_with_qwen(prompt, max_new_tokens=1400)

    return {
        "predicted_sdgs": predicted_sdgs,
        "retrieved_chunks": retrieved_chunks,
        "prompt": prompt,
        "response": response
    }


In [19]:
result = run_full_rag_pipeline(sample_bmc, top_k_sdgs=3)

print("Predicted SDGs:")
for sdg in result["predicted_sdgs"]:
    print("-", sdg)

print("\nGenerated sustainable BMC improvements:")
print(result["response"])


SDG 9: Industry, Innovation and Infrastructure → confidence: 0.8259
SDG 11: Sustainable Cities and Communities → confidence: 0.0614
SDG 12: Responsible Consumption and Production → confidence: 0.0289
Predicted SDGs:
- SDG 9: Industry, Innovation and Infrastructure
- SDG 11: Sustainable Cities and Communities
- SDG 12: Responsible Consumption and Production

Generated sustainable BMC improvements:
### BLOCK 1: Customer Segments
**Current weakness:** The current customer segments focus primarily on small online stores and local retailers without explicitly targeting sustainability-oriented users.
**Sustainable improvement:** **Align offerings with sustainability requirements**, ensuring that all products meet specific eco-friendly standards or certifications like B Corp, Fair Trade, or Rainforest Alliance.
**Retrieved knowledge used:** sustainable_bmc.txt
**Expected environmental impact:** Increased sales among environmentally conscious customers, potentially doubling revenue due to high